In [2]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import os
import json
import time
import shutil
from datetime import datetime

## install reddit API and doing a test
### pip install praw


In [ ]:
import praw

# Replace these with your app credentials
reddit = praw.Reddit(
    client_id="yours",
    client_secret="yours",
    user_agent="my_reddit_scraper_v1"
)

# Test it
print(reddit.read_only) 


True


In [ ]:
# --- Setup ---
subreddits = [
    "popheads", "Music", "BillieEilish", "TaylorSwift", "ArianaGrande",
    "OliviaRodrigo", "SabrinaCarpenter", "TateMcRae", "PopCulture", "popculturechat",
    "Billboard", "Entertainment", "teenagers", 
    "MusicNews", "musictheory", "indieheads", "Fauxmoi",
    "LetsTalkMusic", "NewMusic", "Kpop",
    "YouTubeMusic", "spotify", "musicians", "Songwriters",
    "WeAreTheMusicMakers", "NoStupidQuestions"
]

keywords = [
    # Major Artists
    "Taylor Swift", "Olivia Rodrigo", "Ariana Grande", 
    "Sabrina Carpenter", "Tate McRae", "Billie Eilish",

    # Albums (most iconic / recent)
    "Midnights", "1989", "SOUR", "GUTS", "life of a showgirl",
    "Happier Than Ever", "Thank U, Next", "short n sweet","man's best friend ",
    "positions", "evermore", "folklore","hit me hard and soft", 
    "when we fall asleep where do we go","Eternal Sunshine","Reputation",

    #  Hit Songs
    "Anti-Hero", "Good 4 U", "Vampire", "Bad Guy", 
    "Deja Vu", "Espresso", "Greedy", "What Was I Made For?","birds of a feather",
    "please please please","7 rings","drivers license","shake it off",
    "you should see me in a crown", "positions","you broke me first",

    # Awards & Charts
    "Billboard Hot 100", "Grammy", "MTV VMA", 
    "American Music Awards", "Spotify charts", "Top 40",

    #  Pop culture & fandom
    "pop music", "pop hits", "concert", "tour", "eras tour",
    "fan", "stan", "fandom", "music video", "collab", "remix",

    #  Sentiment / buzzwords
    "bop", "flop", "iconic", "underrated", 
    "viral", "trend", "Spotify Wrapped", "release date"
]

fields = [
    "id", "title", "selftext", "author", "created_utc", "score",
    "num_comments", "subreddit", "keyword", "source"
]

master_file = "reddit_artist_posts.csv"

if os.path.exists(master_file):
    master_df = pd.read_csv(master_file)
else:
    master_df = pd.DataFrame(columns=fields)

batch = []
save_every = 100

# --- Main Loop ---
for subreddit_name in subreddits:
    for keyword in keywords:
        print(f"\n--- Searching '{keyword}' in r/{subreddit_name} ---")
        count = 0
        try:
            print("  [SEARCH] Collecting search results...")
            for submission in reddit.subreddit(subreddit_name).search(keyword, limit=1000):
                batch.append([
                    submission.id,
                    getattr(submission, 'title', ''),
                    getattr(submission, 'selftext', ''),
                    str(submission.author),
                    getattr(submission, 'created_utc', ''),
                    getattr(submission, 'score', ''),
                    getattr(submission, 'num_comments', ''),
                    submission.subreddit.display_name,
                    keyword,
                    "search"  
                ])
                count += 1

            print("  [TOP] Collecting top posts (year)...")
            for submission in reddit.subreddit(subreddit_name).top(time_filter="year", limit=1000):
                if keyword.lower() in submission.title.lower() or keyword.lower() in str(submission.selftext).lower():
                    batch.append([
                        submission.id,
                        getattr(submission, 'title', ''),
                        getattr(submission, 'selftext', ''),
                        str(submission.author),
                        getattr(submission, 'created_utc', ''),
                        getattr(submission, 'score', ''),
                        getattr(submission, 'num_comments', ''),
                        submission.subreddit.display_name,
                        keyword,
                        "top"
                    ])
                    count += 1

            print("  [HOT] Collecting hot posts...")
            for submission in reddit.subreddit(subreddit_name).hot(limit=1000):
                if keyword.lower() in submission.title.lower() or keyword.lower() in str(submission.selftext).lower():
                    batch.append([
                        submission.id,
                        getattr(submission, 'title', ''),
                        getattr(submission, 'selftext', ''),
                        str(submission.author),
                        getattr(submission, 'created_utc', ''),
                        getattr(submission, 'score', ''),
                        getattr(submission, 'num_comments', ''),
                        submission.subreddit.display_name,
                        keyword,
                        "hot" 
                    ])
                    count += 1

            if len(batch) >= save_every:
                temp_df = pd.DataFrame(batch, columns=fields)
                master_df = pd.concat([master_df, temp_df], ignore_index=True)
                master_df.drop_duplicates(subset="id", inplace=True)
                master_df.to_csv(master_file, index=False)
                print(f"Saved {len(master_df)} posts so far...")
                shutil.copy(master_file, "reddit_artist_posts_backup.csv")  # Backup file
                batch = []

            print(f"Finished '{keyword}' in r/{subreddit_name}: {count} posts scraped.")
        except Exception as e:
            print(f"Error: {e}")

# --- Final Save ---
if batch:
    temp_df = pd.DataFrame(batch, columns=fields)
    master_df = pd.concat([master_df, temp_df], ignore_index=True)
    master_df.drop_duplicates(subset="id", inplace=True)
    master_df.to_csv(master_file, index=False)
    print(f"Final save: {len(master_df)} posts in total.")

print("All scraping done. Data saved in reddit_artist_posts.csv")

In [8]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 124438 entries, 0 to 124449
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            124438 non-null  object 
 1   title         124438 non-null  object 
 2   selftext      83270 non-null   object 
 3   author        124310 non-null  object 
 4   created_utc   124438 non-null  float64
 5   score         124438 non-null  int64  
 6   num_comments  124438 non-null  int64  
 7   subreddit     124438 non-null  object 
 8   keyword       124438 non-null  object 
 9   source        124438 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 10.4+ MB
